# Exercise 9.2: GNN on the JetNet dataset revised

From *Programming in High Energy Particle Physics*, Chapter 9

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch09/ex09_2_solution.ipynb)

Exercise 9.2 GNN on the JetNet dataset revised Implement a simple Graph Neural Network using PyTorch Geometric for jet tagging. Use the public JetNet dataset and compare GNN performance with a fully connected baseline.

<details><summary>Hint</summary>

JetNet (R. Kansal et al., 2021) contains gluon, light-quark, top, W and Z jets with up to 30 (or 150) constituents each, stored as relative \(\eta\), relative \(\phi\), relative \(p_T\) and a mask. Install the loader with pip install jetnet torch_geometric and download a class with jetnet.datasets.JetNet.getData(jet_type=["g", "t"], data_dir="./jetnet") . Use the mask to drop padded particles, build one graph per jet with torch_geometric.data.Data , and use DynamicEdgeConv followed by global pooling. For the baseline, flatten the padded particle array (or use jet-level features such as mass and number of constituents) and feed it to the network of Listing 9.3 .

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q jetnet torch-geometric scikit-learn matplotlib numpy")

**Data:** the official JetNet loader downloads real public jet data from Zenodo. The first download can be large; training uses only 600 jets per class by default. The graph removes all padded constituents using the mask.

### Step 1: Load real JetNet gluon and top jets

In [ ]:
import numpy as np, torch
from jetnet.datasets import JetNet
torch.manual_seed(42); rng=np.random.default_rng(42); torch.set_num_threads(2)
N_PER_CLASS=600  # increase after the first pass; download is much larger than this subset
parts=[]; labels=[]
for jet_type,label in [('g',0),('t',1)]:
    p,_=JetNet.getData(jet_type=jet_type,data_dir='./jetnet',num_particles=30,
                       particle_features=['etarel','phirel','ptrel','mask'],jet_features=None,download=True)
    parts.append(p[:N_PER_CLASS].astype('float32')); labels.extend([label]*min(N_PER_CLASS,len(p)))
particles=np.concatenate(parts); y=np.asarray(labels,dtype=np.int64)
assert particles.shape[1:] == (30,4) and len(particles)==len(y)
print('Downloaded JetNet; subset:',particles.shape)

### Step 2: Build masked particle graphs and a dense baseline

In [ ]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import EdgeConv, global_max_pool
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
idx_tr,idx_te=train_test_split(np.arange(len(y)),test_size=.3,stratify=y,random_state=42)
def graph(i,k=6):
    a=particles[i]; x=torch.tensor(a[a[:,3]>.5,:3],dtype=torch.float32)
    if len(x)<2: x=torch.zeros((2,3))
    dist=torch.cdist(x[:,:2],x[:,:2]); dist.fill_diagonal_(float('inf'))
    near=dist.topk(min(k,len(x)-1),largest=False).indices
    dst=torch.arange(len(x)).repeat_interleave(near.shape[1]); src=near.reshape(-1)
    return Data(x=x,edge_index=torch.stack([src,dst]),y=torch.tensor(y[i]))
graphs=[graph(i) for i in range(len(y))]
class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv=EdgeConv(nn.Sequential(nn.Linear(6,32),nn.ReLU(),nn.Linear(32,32)),aggr='max')
        self.out=nn.Linear(32,1)
    def forward(self,b): return self.out(global_max_pool(self.conv(b.x,b.edge_index).relu(),b.batch)).flatten()
flat=particles[:,:,:3].copy(); flat[particles[:,:,3]<.5]=0
flat=flat.reshape(len(y),-1)
scaler=StandardScaler().fit(flat[idx_tr]); flat=torch.tensor(scaler.transform(flat),dtype=torch.float32)
dense=nn.Sequential(nn.Linear(flat.shape[1],64),nn.ReLU(),nn.Linear(64,1))
gnn=GNN()

### Step 3: Train both models and compare held-out ROC curves

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
def fit_dense():
    opt=torch.optim.Adam(dense.parameters(),lr=.003); lossfn=nn.BCEWithLogitsLoss()
    for epoch in range(8):
        for batch in np.array_split(rng.permutation(idx_tr),max(1,len(idx_tr)//64)):
            logits=dense(flat[batch]); loss=lossfn(logits,torch.tensor(y[batch],dtype=torch.float32))
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): return dense(flat[idx_te]).sigmoid().numpy()
def fit_gnn():
    loader=DataLoader([graphs[i] for i in idx_tr],batch_size=64,shuffle=True)
    opt=torch.optim.Adam(gnn.parameters(),lr=.003); lossfn=nn.BCEWithLogitsLoss()
    for epoch in range(8):
        for batch in loader:
            loss=lossfn(gnn(batch),batch.y.float()); opt.zero_grad(); loss.backward(); opt.step()
    gnn.eval(); out=[]
    with torch.no_grad():
        for batch in DataLoader([graphs[i] for i in idx_te],batch_size=64): out.extend(gnn(batch).sigmoid().numpy())
    return np.asarray(out)
sd=fit_dense(); sg=fit_gnn()
for name,score in [('Dense',sd),('PyG EdgeConv',sg)]:
    fpr,tpr,_=roc_curve(y[idx_te],score); print(name,'test AUC',round(roc_auc_score(y[idx_te],score),3)); plt.plot(fpr,tpr,label=name)
plt.plot([0,1],[0,1],'--',color='gray'); plt.xlabel('Gluon mistag rate'); plt.ylabel('Top efficiency'); plt.legend(); plt.show()

The two AUCs use the same held-out jets. This compact PyG implementation recomputes geometric nearest-neighbor edges once per jet and applies EdgeConv; deeper DynamicEdgeConv models can improve performance but need extra PyG compiled extensions. Download time is excluded from the short training runtime.